In [ ]:
import os
from pathlib import Path
from pypdf import PdfReader, PdfWriter

def extract_pdf_documents(
    pdf_path: str, 
    doc_to_extract: list[list[str | int]]
) -> None:
    """
    Extrait des plages de pages d'un PDF source et les enregistre 
    comme de nouveaux documents PDF.

    :param pdf_path: Chemin complet vers le fichier PDF source.
    :param doc_to_extract: Liste de listes où chaque sous-liste contient :
                           [nom_du_nouveau_doc, page_de_début (1-indexée), page_de_fin (1-indexée)].
    """
    
    # Vérifie si le fichier PDF source existe
    if not Path(pdf_path).is_file():
        print(f"Erreur : Le fichier PDF source n'a pas été trouvé à '{pdf_path}'")
        return

    # Utilise Path pour gérer les chemins de manière robuste
    source_file_path = Path(pdf_path)
    # Le dossier de destination est le dossier parent du PDF source
    output_dir = source_file_path.parent

    try:
        # Ouvre le PDF source en mode binaire pour la lecture
        reader = PdfReader(pdf_path)
        
        # Parcourt la liste des documents à extraire
        for doc_name, start_page, end_page in doc_to_extract:
            
            # Crée un nouvel objet PdfWriter pour le nouveau document
            writer = PdfWriter()
            
            # --- Ajustement des index de page ---
            # La numérotation des pages dans les PDF commence à 1 (utilisateur), 
            # mais l'indexation dans pypdf commence à 0 (Python).
            # Les pages à extraire vont de l'index (start_page - 1) 
            # jusqu'à, mais sans inclure, l'index (end_page).
            # Pour inclure la page de fin, nous devons aller jusqu'à l'index (end_page).
            
            # L'extraction inclut la page de début et la page de fin fournies
            # Exemple : pages 2 à 4 (inclus) correspondent aux index 1, 2, 3
            # range(1, 4) donne 1, 2, 3
            start_index = start_page - 1 
            end_index = end_page # range est exclusif sur la borne supérieure
            
            # Vérification des pages pour éviter les erreurs
            max_pages = len(reader.pages)
            if start_page < 1 or end_page > max_pages or start_page > end_page:
                print(f"Avertissement : Plage de pages invalide pour '{doc_name}' ({start_page}-{end_page}). Le PDF a {max_pages} pages. Saut.")
                continue

            # Ajoute les pages sélectionnées au nouveau writer
            for i in range(start_index, end_index):
                writer.add_page(reader.pages[i])
            
            # Construit le chemin du fichier de sortie
            output_filename = f"{doc_name}.pdf"
            output_path = output_dir / output_filename

            # Écrit le nouveau PDF sur le disque
            with open(output_path, "wb") as output_pdf:
                writer.write(output_pdf)
            
            print(f"Document créé : '{output_path}' (Pages {start_page} à {end_page})")

    except Exception as e:
        print(f"Une erreur est survenue lors du traitement du PDF : {e}")


# --- Exemple d'utilisation ---

# 1. Définissez le chemin vers votre fichier PDF source
# REMPLACEZ 'chemin/vers/votre_document.pdf' par le chemin réel de votre fichier
pdf_source_path = 'chemin/vers/votre_document.pdf' 

# 2. Définissez la liste d'extraction
# Format : [ "Nom du fichier de sortie", Page de début (1-indexée), Page de fin (1-indexée) ]
doc_to_extract = [
    ["doc1_chapitre_A", 2, 4],  # Pages 2, 3, 4
    ["doc2_section_B", 5, 6],  # Pages 5, 6
    ["doc3_page_unique", 1, 1], # Page 1 seule
    ["doc4_grand_extrait", 7, 10], # Pages 7, 8, 9, 10
]

# 3. Exécutez la fonction
# Assurez-vous d'avoir un fichier à l'emplacement 'pdf_source_path'
# extract_pdf_documents(pdf_source_path, doc_to_extract)